# u3lib — Python tutorial

Bindings for the header-only C++ **u3lib** (SU(3)/U(3) coupling & recoupling
coefficients). The Python names mirror the C++ API 1:1. Build the module first
with `make python`; the bindings live in [pyu3lib.cpp](pyu3lib.cpp) and build
details in [README](../README.md).

| Python entry point | Purpose |
|---|---|
| `su3.irrep(λ, μ)`, `irrep(λ, μ)`, `irrep(n1, n2, n3)` | SU(3) / U(3) irreps |
| `gelfand(n13,n23,n33,n12,n22,n11)` | Gelfand–Tsetlin state + generator MEs |
| `physicalvector(irrep, L)` | (κ,L) basis state |
| `Rho(a, b, c)` | Outer coupling multiplicity |
| `decomposition(a, b)`, `su3.decomposition(a, b)` | Product irreps with multiplicities |
| `Kappa(lam, mu, L)` | SO(3) branching multiplicity of L in (λ,μ) |
| `physicalBasis`, `gelfandBasis` | basis state enumerators |
| `wigU3U2db().get(...)` | U(3) ⊃ U(2) reduced coefficients (numpy row) |
| `wigU3SO3db().get(...)` / `.get_U3SO3(...)` | U(3) ⊃ SO(3), flat or structured |
| `u3u6db()/u3z6db()/u3u9db().get_U3*(...)` | 6-/9-(λμ) recoupling (structured) |
| `initialize()`, `clear()`, `finalize()` | Session + cache management |

> **Asserts are compiled in** (no `-DNDEBUG`). Indexing an *empty / forbidden*
> coupling, or passing invalid labels, will `abort()` the kernel rather than
> raise a Python exception. Guard with `.empty()` or check `Rho(...)` first.

## 1. Import the extension

Finds `u3lib.so` whether Jupyter is launched from `u3lib` or
`u3lib/python`, and prints the exact path so no installed package shadows it.

In [ ]:
from pathlib import Path
import sys
import numpy as np
from IPython.display import Markdown, display

candidates = []
for base in (Path.cwd(), *Path.cwd().parents):
    candidates.extend((base, base / "python", base / "u3lib" / "python"))
python_dir = next((p.resolve() for p in candidates
                   if (p / "pyu3lib.cpp").is_file()), None)
if python_dir is None:
    raise FileNotFoundError("Launch Jupyter from u3lib or u3lib/python.")

sys.path.insert(0, str(python_dir))
import u3lib as u3
if Path(u3.__file__).resolve().parent != python_dir:
    raise RuntimeError("Another u3lib module is loaded. Restart the kernel.")

np.set_printoptions(precision=10, suppress=True)
print("Python:", sys.version.split()[0])
print("Extension:", u3.__file__)

def table(headers, rows):
    lines = ["| " + " | ".join(map(str, headers)) + " |",
             "| " + " | ".join(["---"] * len(headers)) + " |"]
    lines += ["| " + " | ".join(map(str, r)) + " |" for r in rows]
    display(Markdown("\n".join(lines)))

## 2. Start a session

`initialize()` sets up the utils foundation (Wigner tables + SO(3) caches)
**and** the U(3) coefficient databases. Always pair with `finalize()`.

In [ ]:
u3.initialize()
print("u3lib session initialized.")

## 3. Irreps

- `su3.irrep(λ, μ)` — an SU(3) irrep; `str()` → `(λ,μ)`.
- `irrep(λ, μ)` — a U(3) irrep from SU(3) labels (n33 = 0); `str()` → `[n13,n23,n33]`.
- `irrep(n1, n2, n3)` — a U(3) irrep from the three labels (n1 ≥ n2 ≥ n3).

`u3.irrep` derives from `su3.irrep`. Use `.lam()` / `.lambda_()` (`lambda` is a
Python keyword). Dimension: d(λ,μ) = ½(λ+1)(μ+1)(λ+μ+2).

In [ ]:
labels = [(0, 0), (1, 0), (0, 1), (2, 0), (1, 1), (4, 2)]
rows = []
for lam, mu in labels:
    ir = u3.irrep(lam, mu)
    expected = (lam + 1) * (mu + 1) * (lam + mu + 2) // 2
    assert ir.dim() == expected
    rows.append((f"({lam},{mu})", str(ir), (ir.n13(), ir.n23(), ir.n33()), ir.dim(),
                 round(ir.C1(), 6), round(ir.C2(), 6), round(ir.C3(), 6)))
table(["SU(3) (λ,μ)", "U(3) str", "(n13,n23,n33)", "dim", "C1", "C2", "C3"], rows)

# Two ways to build the same U(3) irrep:
print("irrep(4,2)   ->", str(u3.irrep(4, 2)))          # [6,2,0]
print("irrep(6,4,2) ->", str(u3.irrep(6, 4, 2)),
      "  (lam,mu) =", u3.irrep(6, 4, 2).lam(), u3.irrep(6, 4, 2).mu())

# su3.irrep: Casimirs and conjugation
s = u3.su3.irrep(4, 2)
print("su3.irrep(4,2):", str(s), "| dim", s.dim(),
      "| C2", round(s.C2(), 6), "| C3", round(s.C3(), 6),
      "| conj", str(s.conjugate()))
assert u3.irrep(0, 0).zero()

## 4. Gelfand–Tsetlin states

`gelfand(n13, n23, n33, n12, n22, n11)` — carries the betweenness check, the
SU(2) subgroup labels (ε, 2Λ, 2M_Λ), and the U(3) **generator matrix elements**
`e11`…`e31_22` used when building reduced coefficients.

In [ ]:
g = u3.gelfand(6, 2, 0, 4, 2, 3)
print("state      :", str(g), "| repr", repr(g))
print("valid?     :", g.check())
print("lam, mu    :", g.lambda_(), g.mu())
print("epsilon    :", g.epsilon(), "| 2*Lambda", g.Lambdax2(), "| 2*M_Lambda", g.MLambdax2())
print("e11,e22,e33:", g.e11(), g.e22(), g.e33())
print("e12, e21   :", g.e12(), g.e21())
print("conjugate  :", str(g.conjugate()), "| phase", g.conjugate_phase())

## 5. Outer multiplicity `Rho`

`Rho(a, b, c)` = number of copies of `c` in `a × b`. Two overloads: three
`irrep` objects, or six SU(3) labels.

Spot-checks: in 8⊗8 the octet appears **twice**; the total dimension is 64.
Triality forbids `(4,2)×(4,2)→(4,2)` → **0**.

In [ ]:
adjoint = u3.irrep(1, 1)
channels = [(0, 0), (1, 1), (3, 0), (0, 3), (2, 2), (3, 2)]
rows = []
for lam, mu in channels:
    c = u3.irrep(lam, mu)
    rmax = u3.Rho(adjoint, adjoint, c)
    rows.append((f"({lam},{mu})", c.dim(), rmax, c.dim() * rmax))
table(["output (λ,μ)", "dim", "multiplicity", "dim × mult"], rows)
assert u3.Rho(adjoint, adjoint, adjoint) == 2
assert sum(r[-1] for r in rows) == adjoint.dim() ** 2 == 64
print("8 x 8: octet multiplicity =", u3.Rho(1,1, 1,1, 1,1), "; dim check 64 = 8*8 OK")

print("Rho (2,0)x(2,0)->(4,0):", u3.Rho(2,0, 2,0, 4,0))
print("Rho (4,2)x(4,2)->(4,2):", u3.Rho(4,2, 4,2, 4,2))

## 6. Tensor-product decomposition

`decomposition(a, b)` returns `(irreps, rhomaxs)`, with each output U(3)
irrep paired with its SU(3) outer multiplicity. `su3.decomposition` provides
the same operation on SU(3) irreps directly. For the adjoint product
`8 × 8`, the octet occurs twice; the dimension-weighted sum must be 64.


In [ ]:
adjoint = u3.irrep(1, 1)
outputs, rhomaxs = u3.decomposition(adjoint, adjoint)
channels = [(ir.lam(), ir.mu(), rho)
            for ir, rho in zip(outputs, rhomaxs)]
assert len(channels) == len(set((lam, mu) for lam, mu, _ in channels))
assert (1, 1, 2) in channels
assert sum(ir.dim() * rho for ir, rho in zip(outputs, rhomaxs)) == adjoint.dim() ** 2
print("8 x 8 output channels (lambda, mu, Rho):", channels)

su3_outputs, su3_rhomaxs = u3.su3.decomposition(
    u3.su3.irrep(1, 1), u3.su3.irrep(1, 1))
su3_channels = [(ir.lam(), ir.mu(), rho)
                for ir, rho in zip(su3_outputs, su3_rhomaxs)]
assert sorted(su3_channels) == sorted(channels)
print("SU(3) namespace and U(3) entry points agree.")

## 7. SO(3) branching `Kappa` and basis enumerators

`Kappa(lam, mu, L)` (also `Kappa(irrep, L)`) = the SO(3) branching multiplicity
κ_max: how many independent states of angular momentum `L` the irrep contains.
`physicalBasis(irrep)` lists the physical `(kappa, L)` states; `gelfandBasis(irrep)`
lists the Gelfand states spanning the irrep.

Spot-checks: `(2,0)` contains `L = 0, 2` (κ = 1 each) and no `L = 1`; in `(4,2)`
both `L = 2` and `L = 4` appear **twice** (κ = 2).

In [ ]:
# kappa over L for a couple of irreps:
for lam, mu in [(2, 0), (4, 2)]:
    ir = u3.irrep(lam, mu)
    Lmax = ir.n13() - ir.n33()
    print(f"({lam},{mu}): kappa[L=0..{Lmax}] =", [u3.Kappa(ir, L) for L in range(Lmax + 1)])
assert u3.Kappa(2, 0, 1) == 0 and u3.Kappa(2, 0, 2) == 1
assert u3.Kappa(4, 2, 2) == 2 and u3.Kappa(u3.irrep(4, 2), 2) == 2   # labels == object

print("Number of states for (2,0) = ", u3.irrep(2, 0).dim())

# physicalBasis: the (kappa, L) content of an irrep
pb = u3.physicalBasis(u3.irrep(4, 2))
table(["state", "L", "kappa"], [(repr(v), v.L(), v.kappa()) for v in pb])

# gelfandBasis size == dim of the irrep
gb = u3.gelfandBasis(u3.irrep(2, 0))
table(["state"], [(repr(g),) for g in gb])

## 8. U(3) ⊃ U(2) coefficients — `wigU3U2db`

Per irrep a/b/c, `get(...)` takes `(n13, n23, n33, n12, n22)` and returns the
row over ρ as a 1-D numpy array (index 0 ↔ ρ=1). Interlacing:
n13 ≥ n12 ≥ n23 ≥ n22 ≥ n33.

The example uses the same labels as the C++ U(3) coupling test — each irrep is
`[4,2,0]`, i.e. (λ,μ)=(2,2), with ρ_max = 3.

In [ ]:
state_a = (4, 2, 0, 4, 2)
state_b = (4, 2, 0, 4, 0)
state_c = (4, 2, 0, 4, 2)
for n13, n23, n33, n12, n22 in (state_a, state_b, state_c):
    assert n13 >= n12 >= n23 >= n22 >= n33

u2_db = u3.wigU3U2db()
u2_row = np.asarray(u2_db.get(*state_a, *state_b, *state_c))
assert u2_row.shape == (u3.Rho(2,2, 2,2, 2,2),)
table(["ρ", "reduced coefficient"],
      [(rho, f"{v:+.10f}") for rho, v in enumerate(u2_row, start=1)])

## 9. U(3) ⊃ SO(3) coefficients — `wigU3SO3db`

Two readouts:
- `get(lam,mu,L, ...)` → flat numpy array.
- `get_U3SO3(...)` → a **U3SO3** object indexed as `obj(kappaA, kappaB, kappaC, rho)`,
  with shape getters `kappaAmax()/kappaBmax()/kappaCmax()/rhomax()` and `.data()`.

Spot-check: `(2,0)L2 × (2,0)L2 → (4,0)L2` gives `−√(2/9) ≈ −0.4714`.

In [ ]:
so3_db = u3.wigU3SO3db()
so3 = so3_db.get_U3SO3(2,0,2,  2,0,2,  4,0,2)
print("empty? ", so3.empty(),
      "| shape (kA,kB,kC,rho) =", so3.kappaAmax(), so3.kappaBmax(), so3.kappaCmax(), so3.rhomax())
print(".data() =", so3.data())
print("obj(0,0,0,0) =", so3(0, 0, 0, 0), "  check -sqrt(2/9) =", -np.sqrt(2/9))
np.testing.assert_allclose(so3(0,0,0,0), -np.sqrt(2/9), atol=1e-10)

# A forbidden channel returns an empty object (guard before indexing):
forb = so3_db.get_U3SO3(2,0,0, 2,0,0, 4,0,2)
print("forbidden channel empty? ", forb.empty())

## 10. A normalization check you can reproduce

For `(2,0)⊗(2,0)→(4,0)` every channel is multiplicity-free. For each output
`L_C`, the sum of squared reduced coefficients over the input pairs `(L_A,L_B)`
should be **1**.

In [ ]:
from itertools import product
assert u3.Rho(2,0, 2,0, 4,0) == 1
coeff_rows, norms = [], []
for lc in (0, 2, 4):
    ns = 0.0
    for la, lb in product((0, 2), repeat=2):
        v = np.asarray(so3_db.get(2,0,la, 2,0,lb, 4,0,lc))
        if v.size == 0:
            continue
        coeff_rows.append((la, lb, lc, f"{float(v[0]):+.10f}"))
        ns += float(v[0]) ** 2
    norms.append((lc, ns))
table(["L_A", "L_B", "L_C", "reduced coefficient"], coeff_rows)
table(["L_C", "Σ coeff²"], [(lc, f"{n:.12f}") for lc, n in norms])
np.testing.assert_allclose([n for _, n in norms], np.ones(3), atol=1e-10)
print("All three normalization checks passed.")

## 11. Recoupling — `u3u6db` / `u3z6db` / `u3u9db`

The 6-(λμ) U- and Z-coefficients and the 9-(λμ) coefficients. Each `get_U3*`
takes the relevant coupling **multiplicities** (from `Rho`), mirroring the C++
signature, and returns a structured **U3U6 / U3Z6 / U3U9** object with
multiplicity-indexed `__call__` and `.data()`.

In [ ]:
A = u3.irrep(2,0); B = u3.irrep(0,2); AB = u3.irrep(1,1)
C = u3.irrep(2,1); ABC = u3.irrep(2,1); BC = u3.irrep(1,2)
rAB, rAB_C = u3.Rho(A,B,AB), u3.Rho(AB,C,ABC)
rBC, rA_BC = u3.Rho(B,C,BC), u3.Rho(A,BC,ABC)
u6 = u3.u3u6db().get_U3U6(A, B, ABC, C, AB, rAB, rAB_C, BC, rBC, rA_BC)
print("U3U6 size", u6.size(), "| (0,0,0,0) =", u6(0,0,0,0))
print(u6)

# Z-coefficient: recouple (A x B), (AB x C) vs (A x C), (AC x B). Scalar B.
AC = u3.irrep(2,2)
rAC, rAC_B = u3.Rho(A,C,AC), u3.Rho(AC,B,ABC)
z6 = u3.u3z6db().get_U3Z6(B, A, ABC, C, AB, rAB, rAB_C, AC, rAC, rAC_B)
print("U3Z6 size", z6.size(), "| .data() =", z6.data())

In [ ]:
# 9-(λμ): four legs A,B,C,D; scalar B,C keep every multiplicity 1.
A = u3.irrep(2,0); B = u3.irrep(0,0); C = u3.irrep(0,0); D = u3.irrep(2,0)
AB = u3.irrep(2,0); CD = u3.irrep(2,0); AC = u3.irrep(2,0); BD = u3.irrep(2,0)
ABCD = u3.irrep(4,0)
rAB, rCD = u3.Rho(A,B,AB), u3.Rho(C,D,CD)
rAC, rBD = u3.Rho(A,C,AC), u3.Rho(B,D,BD)
rABCD, rACBD = u3.Rho(AB,CD,ABCD), u3.Rho(AC,BD,ABCD)
u9 = u3.u3u9db().get_U3U9(A,B,C,D, AB,CD, AC,BD, ABCD,
                         rAC,rBD,rABCD, rAB,rCD,rACBD)
print("U3U9 size", u9.size(), "| (0,0,0,0,0,0) =", u9(0,0,0,0,0,0), "| .data()", u9.data())
print(u9)

## 12. Physical generators and the physical→canonical transform

The physical SO(3)-reduced generator RMEs `RME_L`/`RME_Q` and the
physical→canonical expansion `phys_to_cano` are exposed directly.
`RME_L(bra, ket)` returns `(matrix, bra_kappa, ket_kappa)`;
`phys_to_cano(p, M, gArr)` returns `(canoBasis, T)` (column `k` is the
`kappa = k+1` copy).

In [ ]:
import math

b20 = u3.physicalBasis(u3.irrep(2, 0))        # L=2 (index 0), L=0 (index 1)

# L is diagonal in L: <L2||L||L2> = sqrt(L(L+1)(2L+1)) = sqrt(30)
rme_L, bra_kappa, ket_kappa = u3.RME_L(b20[0], b20[0])
print("<L2||L||L2> =", rme_L[0, 0], "  sqrt(30) =", math.sqrt(30))

# quadrupole Q: m = L'-L; diagonal (m=0) and m=2 branches
print("<L2||Q||L2> =", u3.RME_Q(b20[0], b20[0])[0][0, 0])
print("<L2||Q||L0> =", u3.RME_Q(b20[0], b20[1])[0][0, 0])

# physical -> canonical expansion of |(2,0) L=2> at M=0
ir = u3.irrep(2, 0)
gArr = u3.gelfandBasis(ir)
p = u3.physicalvector(ir, 2)
canoBasis, T = u3.phys_to_cano(p, 0, gArr)
print("M=0 amplitudes:", T.ravel(), " columns orthonormal:", float((T.T @ T)[0, 0]))

## 13. Cache behavior

The databases cache computed coefficients. `clear()` discards cached values
while keeping the session initialized; the next request recomputes them.

In [ ]:
ref = np.array(u2_db.get(*state_a, *state_b, *state_c), copy=True)
u3.clear()
again = np.asarray(u2_db.get(*state_a, *state_b, *state_c))
np.testing.assert_allclose(again, ref, atol=1e-10)
print("Repeated and recomputed results agree.")

## 14. Parallelism

`u3lib` is **not** parallelizable from Python threads. The extension holds the
GIL (so threads give no speed-up), and the C++ coefficient databases keep
per-thread state as OpenMP `threadprivate` storage keyed on
`omp_get_thread_num()`. A `threading` / `ThreadPoolExecutor` worker is a plain OS
thread, **not** an OpenMP thread, so calling `initialize_thread()` there aborts
the interpreter. Do not use Python threads with `u3lib`.

**Native parallelism is C++ / OpenMP.** Inside an OpenMP parallel region each
thread sets up and tears down its own database state; the `src/` example drivers
show the pattern:

```cpp
u3::initialize(&argc, &argv);            // once, on the main thread
#pragma omp parallel num_threads(4)
{
    u3::initialize_thread();             // per OpenMP thread
#pragma omp for
    for (int t = 0; t < work.size(); ++t) {
        auto blk = u3::wigU3SO3db().get_U3SO3(/* labels */);
        // use blk; guard shared I/O with #pragma omp critical
    }
    u3::finalize_thread();
}
u3::finalize();
```

**From Python, parallelize across _processes_, not threads** — each process has
its own interpreter, GIL and library state. Run it as a standalone script (in a
notebook, `spawn` cannot pickle a cell-defined worker, so this does not work in a
cell):

```python
# save as run_parallel.py, then: python run_parallel.py
import concurrent.futures as cf, multiprocessing as mp

def work(idx):
    import u3lib as u3
    u3.initialize()
    try:
        return idx, u3.wigU3SO3db().get_U3SO3(2, 0, 2, 2, 0, 2, 4, 0, 2)(0, 0, 0, 0)
    finally:
        u3.finalize()

if __name__ == "__main__":
    ctx = mp.get_context("spawn")
    with cf.ProcessPoolExecutor(max_workers=4, mp_context=ctx) as ex:
        print([f.result() for f in [ex.submit(work, i) for i in range(4)]])
```

## 15. Finish

Finalize after all calculations. Don't use the database objects afterward; to
run more coefficient cells, rerun the initialization cell (or restart & run all).

In [ ]:
u3.finalize()
print("Session finalized. All tutorial checks passed.")